# Home Care Cost Model — Canadian Service Mix Analysis

Companion notebook to the working paper *The Home Care Cost Model: Personal Support, Housekeeping, and Service Mix Decisions for Aging in Place in Canada* (Cook, 2026).

This notebook loads the published datasets and reproduces the three headline findings from the paper:

1. The hybrid service mix (PSW + housekeeping) is typically 10–20 percent cheaper than an all-PSW plan.
2. Tax credits reduce eligible households' out-of-pocket by 15–35 percent, with substantial cross-province variation.
3. The gap between subsidised hours and model-recommended hours varies by a factor of three across jurisdictions.

All data is deterministic and under CC BY 4.0. Code is MIT-licensed. This is a reference framework, not clinical or financial advice.

In [ ]:
import pandas as pd
import numpy as np

scenarios = pd.read_csv('home_care_scenarios.csv')
archetypes = pd.read_csv('home_care_cost_model_archetypes.csv')
subsidy_gap = pd.read_csv('home_care_subsidy_gap.csv')
tax_sensitivity = pd.read_csv('home_care_tax_relief_sensitivity.csv')
services = pd.read_csv('home_care_services_canada.csv')

print(f'Scenarios: {len(scenarios):,} rows')
print(f'Archetypes: {len(archetypes):,} rows')
print(f'Subsidy gap: {len(subsidy_gap):,} rows')
print(f'Tax sensitivity: {len(tax_sensitivity):,} rows')
print(f'Services reference: {len(services):,} rows')

## Finding 1 — Hybrid savings vs all-PSW

For each scenario where housekeeping hours are positive, compute the hybrid savings as a fraction of the all-PSW counterfactual.

In [ ]:
hybrid = scenarios[scenarios['recommended_housekeeping_hours_per_week'] > 0].copy()
hybrid['savings_pct'] = (
    hybrid['hybrid_savings_vs_all_psw_monthly_cad'] /
    hybrid['all_psw_cost_comparison_monthly_cad']
)
print('Hybrid savings as fraction of all-PSW counterfactual:')
print(hybrid['savings_pct'].describe())

## Finding 2 — Tax relief effectiveness by income band

In [ ]:
tax_summary = tax_sensitivity.groupby('net_family_income_cad').agg({
    'effective_credit_fraction_of_oop': ['mean', 'std'],
    'total_credits_value_cad': 'mean'
}).round(3)
print(tax_summary)

## Finding 3 — Cross-province subsidy gap

In [ ]:
gap_by_province = subsidy_gap.groupby('jurisdiction_code').agg({
    'dollar_gap_monthly_cad': 'mean',
    'hours_gap_per_week': 'mean'
}).round(2).sort_values('dollar_gap_monthly_cad', ascending=False)
print('Average monthly dollar gap by jurisdiction:')
print(gap_by_province)

## Disclaimer

Reference model only. Not clinical or financial advice; consult a regulated health professional or registered tax practitioner for individual decisions. Synthetic scenarios are deterministic from seed 42 and must not be cited as empirical observations.